<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Curso_python_Ciencia_de_Datos/blob/main/d)_Exercise_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En este ejercicio, utilizarás **pipelines** para mejorar la eficiencia de tu código de aprendizaje automático.

### Configuración

Las preguntas a continuación te darán retroalimentación sobre tu trabajo. Ejecuta la siguiente celda para configurar el sistema de retroalimentación.

In [ ]:
# Carga las librerias necesarias
import pandas as pd
import os
import kagglehub
from sklearn.model_selection import train_test_split

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nickptaylor/iowa-house-prices")

print("Path to dataset files:", path)

train_csv_path = path + '/train.csv'
test_csv_path = path + '/test.csv'

X_full = pd.read_csv(train_csv_path, index_col='Id')
X_test_full = pd.read_csv(test_csv_path, index_col='Id')

In [ ]:
# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# Break off validation set from training data
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_full, y,
                                                                train_size=0.8, test_size=0.2,
                                                                random_state=0)

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_cols = [cname for cname in X_train_full.columns if
                    X_train_full[cname].nunique() < 10 and
                    X_train_full[cname].dtype == "object"]

# Select numerical columns
numerical_cols = [cname for cname in X_train_full.columns if
                X_train_full[cname].dtype in ['int64', 'float64']]

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

In [ ]:
X_train.head()

La siguiente celda de código utiliza código del tutorial para preprocesar los datos y entrenar un modelo. Ejecuta este código sin cambios.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define model
model = RandomForestRegressor(n_estimators=100, random_state=0)

# Bundle preprocessing and modeling code in a pipeline
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)
                     ])

# Preprocessing of training data, fit model
clf.fit(X_train, y_train)

# Preprocessing of validation data, get predictions
preds = clf.predict(X_valid)

print('MAE:', mean_absolute_error(y_valid, preds))

El código arroja un valor de aproximadamente 17862 para el error absoluto medio (MAE). En el siguiente paso, modificarás el código para mejorar los resultados.

### **Paso 1:** Mejorar el rendimiento

### Parte A

¡Ahora te toca a ti! En la celda de código a continuación, define tus propios pasos de preprocesamiento y un modelo de bosque aleatorio. Completa los valores de las siguientes variables:
- `numerical_transformer`
- `categorical_transformer`
- `model`

Para aprobar esta parte del ejercicio, solo necesitas definir pasos de preprocesamiento válidos y un modelo de bosque aleatorio.

In [ ]:
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant') # Your code here

# Preprocessing for categorical data
# Your code here
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define model
# Your code here
model = RandomForestRegressor(n_estimators=100, random_state=0)

### **Parte B**

Ejecute la celda de código a continuación sin realizar cambios.

Para superar este paso, debe haber definido una canalización en **Parte A** que consiga un MAE menor que el código anterior. Se recomienda que se tome su tiempo aquí y pruebe diferentes enfoques para ver qué tan bajo puede conseguir el MAE. (_Si su código no pasa, por favor, modifique los pasos de preprocesamiento y el modelo en la Parte A._).

In [ ]:
# Código de preprocesamiento y modelado en un pipeline
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('model', model)])

# Preprocesamiento de los datos de entrenamiento, ajuste del modelo
my_pipeline.fit(X_train, y_train)

# Preprocesamiento de los datos de validación, obtención de predicciones
preds = mi_pipeline.predict(X_valid)

# Evaluar el modelo
score = mean_absolute_error(y_valid, preds)
print('MAE:', score)

### **Paso 2:** Generar predicciones de prueba
Ahora, utilizarás tu modelo entrenado para generar predicciones con los datos de prueba.

In [ ]:
test_preds = my_pipeline.predict(X_test)

Ejecuta la siguiente celda de código sin cambios para guardar tus resultados en un archivo CSV que podrá descargar en el icono de la carpeta, en la parte izquierda de tu colab.

In [ ]:
# Guardar predicciones de prueba en un archivo
predicciones = pd.DataFrame({'Id': X_test.index,
                              'PrecioVenta': test_preds})
predicciones.to_csv('predicciones_pipeline.csv', index=False)

Si deseas seguir trabajando para mejorar tu rendimiento, puedes modificar tu código y repetir el proceso. Hay mucho margen de mejora a medida que avances y aprendas.